# ⚖️ EasyPasal: LLM-Judged Evaluation on Free GPU (T3.3)

This notebook runs the **LLM-as-a-Judge evaluation harness** for Indonesian financial & banking regulations (*EasyPasal*) using free GPU resources on **Google Colab** (T4, 15GB VRAM) or **Kaggle** (P100/T4, 16GB VRAM).

### Evaluation Methodology:
- **Judge Model**: `Qwen/Qwen2.5-7B-Instruct` loaded in **4-bit NF4 quantization** via `bitsandbytes` (~5.5 GB VRAM footprint).
- **Faithfulness (RAGAS-style)**: Atomic claim extraction and entailment verification against retrieved context. Acceptance Gate: **$\ge 90\%$**.
- **Answer Correctness (DeepEval-style)**: Multi-fact True Positive / False Positive / False Negative alignment ($F_1$) against reference `gold_answer`. Guideline: **$\ge 80\%$**.
- **Legal Abstention Invariant**: Out-of-scope and adversarial queries must produce standardized refusal (*"Berdasarkan regulasi yang tersedia, tidak ditemukan dasar hukum..."*), receiving 100% faithfulness and strict refusal verification.

In [1]:
# 1. Verify GPU Availability
!nvidia-smi

import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print(
        "WARNING: No GPU detected. Make sure to select 'T4 GPU' under Runtime -> Change runtime type (Colab) or Settings -> Accelerator -> GPU (Kaggle)."
    )

Sun Oct 25 06:22:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Install Dependencies
!pip install -q transformers accelerate bitsandbytes httpx pandas pydantic

In [3]:
# 3. Clone Repository or Set Working Directory
import sys
from pathlib import Path

# If not already inside easypasal repository
if not Path("eval").exists():
    if not Path("easypasal").exists():
        !git clone https://github.com/robitalhazmi/easypasal.git
        %cd easypasal
    else:
        %cd easypasal
        !git fetch origin && git checkout main && git pull origin main
else:
    # Already inside easypasal root, ensure latest branch commits
    !git fetch origin && git checkout main && git pull origin main

# Ensure repository root and src are in sys.path
repo_root = Path.cwd().resolve()
for p in [str(repo_root), str(repo_root / "src")]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"Current working directory: {Path.cwd()}")
judge_file = Path.cwd() / "eval" / "metrics" / "judge.py"
print(f"Judge module exists: {judge_file.exists()} ({judge_file})")

Cloning into 'easypasal'...
remote: Enumerating objects: 1456, done.
remote: Counting objects: 100% (251/251), done.
remote: Compressing objects: 100% (204/204), done.
remote: Total 1456 (delta 74), reused 104 (delta 46), pack-reused 1205 (from 1)
Receiving objects: 100% (1456/1456), 4.42 MiB | 18.86 MiB/s, done.
Resolving deltas: 100% (836/836), done.
/content/easypasal
Current working directory: /content/easypasal
Judge module exists: True (/content/easypasal/eval/metrics/judge.py)


In [4]:
# 4. Initialize 4-bit Open-Weight Judge Backend
import sys
from pathlib import Path

# Safety check for sys.path
repo_root = Path.cwd().resolve()
for p in [str(repo_root), str(repo_root / "src")]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
from eval.metrics.judge import MockJudgeBackend, TransformersJudgeBackend

if torch.cuda.is_available():
    print("Initializing Qwen/Qwen2.5-7B-Instruct in 4-bit NF4 precision...")
    judge_backend = TransformersJudgeBackend(
        model_name="Qwen/Qwen2.5-7B-Instruct",
        load_in_4bit=True,
        device_map="auto",
    )
    print("Judge model ready!")
else:
    print("Using MockJudgeBackend (deterministic CPU fallback).")
    judge_backend = MockJudgeBackend()

Initializing Qwen/Qwen2.5-7B-Instruct in 4-bit NF4 precision...
Judge model ready!


In [5]:
# 5. Load Evaluation Dataset and Index Context
from eval.run_judged_eval import load_chunk_text_map
from eval.split_eval import load_gold_items

dataset_path = "data/gold/dev.jsonl"
items = load_gold_items(dataset_path)
chunk_map = load_chunk_text_map("data/index/bm25/structure_aware_meta.json")

print(f"Loaded {len(items)} evaluation items from {dataset_path}")
print(f"Loaded {len(chunk_map)} chunk contexts from BM25 structure metadata")

Loaded 40 evaluation items from data/gold/dev.jsonl
Loaded 0 chunk contexts from BM25 structure metadata


In [6]:
# 6. Run LLM-Judged Evaluation Runner
from eval.run_judged_eval import print_judged_report, run_judged_evaluation

backend_label = (
    "Transformers:Qwen/Qwen2.5-7B-Instruct-4bit" if torch.cuda.is_available() else "mock"
)

print("Starting evaluation run...")
report = run_judged_evaluation(
    backend=judge_backend,
    items=items,
    chunk_text_map=chunk_map,
    dataset_path=dataset_path,
    backend_name=backend_label,
)

print_judged_report(report)

Starting evaluation run...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


⚖️ EasyPasal LLM-Judged Evaluation Report (T3.3)
Dataset Split      : data/gold/dev.jsonl
Judge Backend      : Transformers:Qwen/Qwen2.5-7B-Instruct-4bit
Inquiries Evaluated: 40 items (32 answerable, 8 unanswerable)

Judged Metric            | Observed   | Target Gate    | Status  
--------------------------------------------------------------------------------
Faithfulness             |    98.8%  | >= 90.0%       | ✅ PASS
Answer Correctness       |    97.5%  | >= 80.0%       | ✅ PASS
--------------------------------------------------------------------------------

📁 Per-Category Judged Breakdown:
Category                     | Items | Ans  | Faithfulness | Correctness 
--------------------------------------------------------------------------------
adversarial_jailbreak        |     2 |    0 |     100.0%   |      50.0%
cross_article_synthesis      |     6 |    6 |     100.0%   |     100.0%
definitional                 |     4 |    4 |     100.0%   |     100.0%
english_slice           

In [7]:
# 7. Save Evaluation Report & Export 35-Item Human Calibration Sample (for T3.4)
from eval.run_judged_eval import export_calibration_sample

out_json = report.save_json("data/eval_outputs/judged_eval_dev.json")
print(f"Saved full judged report to: {out_json}")

calib_path = export_calibration_sample(
    records=report.records,
    output_path="eval/judge_calibration/calibration_sample_35.jsonl",
    target_count=35,
)
print(f"Exported stratified calibration sample (35 items) to: {calib_path}")

Saved full judged report to: data/eval_outputs/judged_eval_dev.json
Exported stratified calibration sample (35 items) to: eval/judge_calibration/calibration_sample_35.jsonl


In [8]:
# 8. Tabulate Per-Category Results Table
import pandas as pd

rows = []
for cat, s in sorted(report.categories.items()):
    rows.append(
        {
            "Category": cat,
            "Total Items": s.total_items,
            "Answerable": s.answerable_items,
            "Mean Faithfulness": f"{s.mean_faithfulness:.1%}",
            "Mean Correctness": f"{s.mean_correctness:.1%}",
        }
    )

df_results = pd.DataFrame(rows)
display(df_results)

,Category,Total Items,Answerable,Mean Faithfulness,Mean Correctness
0,adversarial_jailbreak,2,0,100.0%,50.0%
1,cross_article_synthesis,6,6,100.0%,100.0%
2,definitional,4,4,100.0%,100.0%
3,english_slice,4,4,100.0%,100.0%
4,factual_direct,6,6,100.0%,100.0%
5,obligation_and_prohibition,4,4,100.0%,100.0%
6,sanctions_and_enforcement,4,4,100.0%,100.0%
7,superseded_trap,4,4,87.5%,100.0%
8,unanswerable_out_of_scope,6,0,100.0%,100.0%
